# Choosing and configuring decoders

qLDPC distinguishes two decoding tasks:

- An **error decoder** maps a syndrome to an inferred physical error.
  Code methods such as `get_logical_error_rate_func` and `get_distance_bound` consume error decoders.
- An **observable decoder** compiles against a Stim detector error model, and maps detection events to predicted observable flips.
  Sinter consumes observable decoders, which build an error decoder internally and convert the errors it infers into observable flips.

This notebook shows how to configure built-in error decoders, how to supply a custom decoder of your own, and how to wrap either in an observable decoder for circuit-level simulations.

## Setup

Run the next cell once in a fresh notebook environment.
IPython's `%pip` magic installs into the active kernel, so the following imports use the same environment.

In [1]:
%%capture
%pip install qldpc

In [2]:
import functools
import inspect
import itertools
import os

import numpy as np
import numpy.typing as npt
import sinter
import stim

from qldpc import codes, decoders
from qldpc.objects import Pauli

NUM_WORKERS = max(1, min(4, (os.cpu_count() or 1) - 1))

## Configure a built-in error decoder

Each built-in decoder has a helper function in `qldpc.decoders`:
`bp_osd`, `bp_lsd`, `bf`, `mwpm`, `relay_bp`, `lookup_table`, `ilp`, and `guf`.
A helper returns a `DecoderSpec`, which only stores settings.
The method that consumes a `DecoderSpec` builds a decoder once it knows which parity check matrix or detector error model to decode, so one `DecoderSpec` can configure decoders for many matrices.

In [3]:
bp_lsd_settings = decoders.bp_lsd(max_iter=30, bp_method="minimum_sum")
bp_lsd_settings

decoders.bp_lsd(max_iter=30, bp_method='minimum_sum')

The signature of a helper lists the options of its decoder, together with their default values.
Autocomplete and `help(decoders.bp_lsd)` show the same information, with a description of each option.

In [4]:
for parameter in inspect.signature(decoders.bp_lsd).parameters.values():
    print(parameter)

error_rate: 'float' = 0.001
error_channel: 'npt.NDArray[np.floating] | Sequence[float] | None' = None
max_iter: 'int' = 0
bp_method: "Literal['product_sum', 'minimum_sum', 'ps', 'ms']" = 'product_sum'
ms_scaling_factor: 'float' = 1.0
schedule: "Literal['parallel', 'serial']" = 'parallel'
omp_thread_count: 'int' = 1
random_schedule_seed: 'int' = 0
serial_schedule_order: 'Sequence[int] | None' = None
bits_per_step: 'int' = 1
lsd_method: "Literal['LSD_0', 'LSD_E', 'LSD_CS']" = 'LSD_0'
lsd_order: 'int' = 0
always_run_lsd: 'bool' = False


A misspelled option is rejected immediately, rather than silently ignored by the underlying decoder.
(The exceptions are `relay_bp` and `ilp`, which forward additional options to the Relay-BP decoder class or integer-program solver that they configure.)

In [5]:
try:
    decoders.bp_lsd(lsd_ordr=2)
except TypeError as error:
    print(f"TypeError: {error}")

TypeError: bp_lsd() got an unexpected keyword argument 'lsd_ordr'


## Decode syndromes

`decoders.decode` builds a decoder and decodes one syndrome, while `decoders.get_decoder` returns a decoder that can be reused.
If `decoder` is not provided, qLDPC uses BP+OSD for binary codes and generalized union-find for codes over larger fields.

In [6]:
code = codes.HammingCode(3)
error = code.field.Zeros(len(code))
error[2] = 1
syndrome = code.matrix @ error

print("syndrome:", syndrome)
print("BP+LSD:", decoders.decode(code.matrix, syndrome, decoder=bp_lsd_settings))

error_decoder = decoders.get_decoder(
    code.matrix, decoder=decoders.bp_osd(osd_method="OSD_CS", osd_order=2)
)
print("BP+OSD:", error_decoder.decode(syndrome))

syndrome: [0 1 1]


BP+LSD: [0 0 1 0 0 0 0]
BP+OSD: [0 0 1 0 0 0 0]


## Use decoders in code methods

Code methods accept the same `decoder` argument.
For example, the logical error rate of a surface code in a code-capacity model, decoded with minimum-weight perfect matching (MWPM):

In [7]:
surface_code = codes.SurfaceCode(3)
logical_error_rate = surface_code.get_logical_error_rate_func(
    num_samples=2_000, max_error_rate=0.1, decoder=decoders.mwpm()
)
logical_error_rate(0.01)  # (logical error rate, uncertainty) at a physical error rate of 1%

(0.0015658667385573587, 8.425813113974013e-05)

A CSS code decodes X-type and Z-type errors independently, and each sector can use a different decoder:

- `decoder_x` infers X-type errors from their syndrome with respect to the Z-type stabilizers, and
- `decoder_z` infers Z-type errors from their syndrome with respect to the X-type stabilizers.

A sector that is not configured explicitly falls back to the shared `decoder` argument.

In [8]:
logical_error_rate = surface_code.get_logical_error_rate_func(
    num_samples=2_000,
    max_error_rate=0.1,
    decoder_x=decoders.mwpm(),
    decoder_z=decoders.bp_osd(),
)
logical_error_rate(0.01)

(0.0015820035131734313, 8.43193377013257e-05)

Randomized distance bounds also accept a decoder.
Providing one selects the decoder-based distance-bounding algorithm.

In [9]:
hgp_code = codes.HGPCode(codes.RepetitionCode(3))
hgp_code.get_code_params(bound=20, decoder=decoders.bp_osd(osd_method="OSD_CS", osd_order=2))

(13, 1, 3)

## Supply a custom decoder

Any object with a `decode` method that maps a syndrome to an inferred error is an error decoder, in the sense of the `decoders.ErrorDecoder` protocol.
As an example, the decoder below enumerates low-weight errors when it is built, and decodes each syndrome to a minimum-weight error that is consistent with it.
Its constructor accepts a parity check matrix or a detector error model, like the built-in decoders.

In [10]:
class MinimumWeightDecoder:
    """Decode each syndrome to a minimum-weight error, found by enumerating low-weight errors.

    This decoder assumes binary (GF(2)) arithmetic, and is only practical for small decoding problems.
    """

    def __init__(
        self,
        pcm_or_dem: npt.NDArray[np.int_] | stim.DetectorErrorModel,
        max_weight: int = 2,
    ) -> None:
        if isinstance(pcm_or_dem, stim.DetectorErrorModel):
            matrix = decoders.DetectorErrorModelArrays(pcm_or_dem).detector_flip_matrix.toarray()
        else:
            matrix = np.asarray(pcm_or_dem).view(np.ndarray)
        self.num_errors = matrix.shape[1]

        # enumerate errors in order of increasing weight, keeping the first error for each syndrome
        self.syndrome_to_error: dict[tuple[int, ...], npt.NDArray[np.int_]] = {}
        for weight in range(max_weight + 1):
            for support in itertools.combinations(range(self.num_errors), weight):
                error = np.zeros(self.num_errors, dtype=int)
                error[list(support)] = 1
                syndrome = tuple((matrix @ error % 2).tolist())
                self.syndrome_to_error.setdefault(syndrome, error)

    def decode(self, syndrome: npt.NDArray[np.int_]) -> npt.NDArray[np.int_]:
        """Decode a syndrome to a minimum-weight error, or to no error if none was found."""
        key = tuple(np.asarray(syndrome).view(np.ndarray).astype(int).tolist())
        no_error = np.zeros(self.num_errors, dtype=int)
        return self.syndrome_to_error.get(key, no_error).copy()

A custom decoder can be passed as `decoder=` in three ways:

1. As a **constructor**, such as the class itself, which qLDPC calls with the matrix or detector error model to decode.
2. As any other **callable** that builds a decoder from a matrix or detector error model, which is a convenient way to fix constructor options.
3. As a **prebuilt decoder**, which qLDPC uses as is.

A custom decoder can optionally define a `decode_batch` method to decode many syndromes at once, or set `has_erasure_bit = True` to declare that it appends an erasure flag to each inferred error.
See the [decoder guide](https://qldpc.readthedocs.io/decoders.html) for details.

In [11]:
print("class:", decoders.decode(code.matrix, syndrome, decoder=MinimumWeightDecoder))

build_weight_1_decoder = functools.partial(MinimumWeightDecoder, max_weight=1)
print("callable:", decoders.decode(code.matrix, syndrome, decoder=build_weight_1_decoder))

prebuilt_decoder = MinimumWeightDecoder(code.matrix, max_weight=1)
print("prebuilt:", decoders.decode(code.matrix, syndrome, decoder=prebuilt_decoder))

class: [0 0 1 0 0 0 0]
callable: [0 0 1 0 0 0 0]
prebuilt: [0 0 1 0 0 0 0]


A prebuilt decoder only works for the matrix that it was built for.
The X-sector and Z-sector decoders of a CSS code decode different stabilizer matrices, so prebuilt decoders for a CSS code are passed per sector:

In [12]:
stabilizer_ops_x = surface_code.get_stabilizer_ops(Pauli.X, canonicalized=False)
stabilizer_ops_z = surface_code.get_stabilizer_ops(Pauli.Z, canonicalized=False)
logical_error_rate = surface_code.get_logical_error_rate_func(
    num_samples=2_000,
    max_error_rate=0.1,
    decoder_x=MinimumWeightDecoder(stabilizer_ops_z),
    decoder_z=MinimumWeightDecoder(stabilizer_ops_x),
)
logical_error_rate(0.01)

(0.0015709879463010248, 8.425075397167299e-05)

Some methods decode a matrix that they construct internally.
Randomized distance bounds, for example, decode an effective check matrix that stacks the checks and logical operators of a code.
These methods reject prebuilt decoders, but accept a constructor, which they call with the matrix that they build:

In [13]:
try:
    hgp_code.get_distance_bound(decoder=MinimumWeightDecoder(hgp_code.matrix_z))
except ValueError as error:
    print(f"ValueError: {error}")

hgp_code.get_distance_bound(
    num_trials=5, decoder=functools.partial(MinimumWeightDecoder, max_weight=3)
)

ValueError: A prebuilt decoder cannot be passed as decoder= here because decoder-based distance bounds decode an internal effective check matrix that stacks the checks and logical operators of the code.  Pass decoder settings such as decoder=decoders.bp_osd(...), or a decoder constructor, instead


3

## Predict observable flips in circuit-level simulations

An `ObservableDecoder` wraps error-decoder settings or an error-decoder constructor.
When compiled for a detector error model, it builds the error decoder for that model, and maps the errors that the error decoder infers to observable flips.
The compiled decoder predicts the observable flips of one shot with `decode_observables`, or of many shots with `decode_shots`.

Here, we compare a few error decoders on a repetition-code memory experiment:

In [14]:
circuit = stim.Circuit.generated(
    "repetition_code:memory", distance=5, rounds=10, after_clifford_depolarization=0.01
)
dem = circuit.detector_error_model()
sampler = circuit.compile_detector_sampler(seed=0)
detection_events, observable_flips = sampler.sample(10_000, separate_observables=True)

for name, decoder in [
    ("MWPM", decoders.mwpm()),
    ("BP+OSD", decoders.bp_osd()),
    ("MinimumWeightDecoder", functools.partial(MinimumWeightDecoder, max_weight=2)),
]:
    observable_decoder = decoders.ObservableDecoder(decoder=decoder)
    compiled_decoder = observable_decoder.compile_decoder_for_dem(dem)
    predicted_flips = compiled_decoder.decode_shots(detection_events.astype(np.uint8))
    logical_error_rate = np.mean(np.any(predicted_flips != observable_flips, axis=1))
    print(f"{name}: logical error rate {logical_error_rate:.4f}")

MWPM: logical error rate 0.0003
BP+OSD: logical error rate 0.0004


MinimumWeightDecoder: logical error rate 0.0047


Unlike MWPM and BP+OSD, `MinimumWeightDecoder` ignores the error probabilities of the detector error model, and gives up on syndromes of errors with weight greater than 2, so it performs worse here.

An observable decoder, and window or subgraph decoders that build one error decoder for each window or subgraph, cannot use a prebuilt error decoder: they need decoder settings or a constructor.

In [15]:
try:
    decoders.ObservableDecoder(decoder=MinimumWeightDecoder(dem))
except ValueError as error:
    print(f"ValueError: {error}")

ValueError: A prebuilt decoder cannot be passed as decoder= here because an ObservableDecoder builds a new error decoder for each (simplified) detector error model that it is compiled for.  Pass decoder settings such as decoder=decoders.bp_osd(...), or a decoder constructor, instead


Observable decoders plug into `sinter.collect` as custom decoders.
Below, we compare decoding the whole memory experiment at once against decoding it in sliding windows of four rounds, which commit corrections two rounds at a time.
Sinter samples in separate worker processes, which receive decoders by pickling them.
Decoder specs such as `decoders.mwpm()` pickle cleanly; to use a custom error decoder with multiple workers, define its class in an importable module rather than in a notebook.

In [16]:
custom_decoders = {
    "monolithic": decoders.ObservableDecoder(decoder=decoders.mwpm()),
    "sliding window": decoders.SlidingWindowDecoder(
        window_size=4, stride=2, decoder=decoders.mwpm()
    ),
}
stats = sinter.collect(
    tasks=[sinter.Task(circuit=circuit, decoder=name) for name in custom_decoders],
    custom_decoders=custom_decoders,
    num_workers=NUM_WORKERS,
    max_shots=20_000,
    max_errors=200,
    print_progress=False,
)
for stat in sorted(stats, key=lambda stat: stat.decoder):
    print(f"{stat.decoder}: logical error rate {stat.errors / stat.shots:.4f} ({stat.shots} shots)")

monolithic: logical error rate 0.0004 (20000 shots)
sliding window: logical error rate 0.0007 (20000 shots)


With this number of shots, each decoder makes only a handful of mistakes, so the difference between them is within statistical noise.
In general, a sliding-window decoder trades some accuracy for a bounded decoding latency; see the [sliding-window decoding example](https://qldpc.readthedocs.io/examples/logical_error_rates/4_sliding_window_decoding.html) for a more careful comparison.

## Lookup tables that return observable flips

Most decoders return errors, which an observable decoder converts into observable flips.
A lookup table can also map each syndrome directly to its most likely observable flip, accounting for all low-weight errors that are consistent with the syndrome.
`ObservableLookupDecoder` builds such a table, and predicts observable flips with `decode_observables`.
In contrast, `LookupDecoder` returns an error, and is an error decoder like the others in this notebook.

In [17]:
observable_lookup = decoders.ObservableLookupDecoder(dem, max_weight=2)
num_shots = 1_000
predicted_flips = np.array(
    [observable_lookup.decode_observables(events) for events in detection_events[:num_shots]]
)
logical_error_rate = np.mean(np.any(predicted_flips != observable_flips[:num_shots], axis=1))
print(f"ObservableLookupDecoder: logical error rate {logical_error_rate:.4f}")

ObservableLookupDecoder: logical error rate 0.0020
